In [1]:
# Block 1: Setup, Imports, and GOOD Hyperparameters
# ==============================================================================
import warnings
import time
import pandas as pd
import numpy as np
import gc
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.cluster import MiniBatchKMeans

warnings.filterwarnings('ignore')

# --- Load Data ---
drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm', 'view_otherwater', 'view_other']
DATA_PATH = './'
df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
y_true = df_train['sale_price'].copy()

# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
COMPETITION_ALPHA = 0.1

print("--- Using PROVEN Hyperparameters from winner_v1_301 ---")
# === MEAN MODEL PARAMETERS (FROM WINNER_V1_301) ===
best_params_mean = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'n_jobs': -1, 'random_state': RANDOM_STATE,
    'eta': 0.041599605162930035,
    'max_depth': 8,
    'subsample': 0.8211034324219306,
    'colsample_bytree': 0.8683430739702909,
    'lambda': 3.717655605557664,
    'alpha': 2.8186169330124836e-05
}

# === ERROR MODEL PARAMETERS (FROM WINNER_V1_301) ===
best_params_error = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'n_jobs': -1, 'random_state': RANDOM_STATE,
    'eta': 0.01725756977806232,
    'max_depth': 9,
    'subsample': 0.9325133327284854,
    'colsample_bytree': 0.7391175883075835,
    'lambda': 0.5897279418593165,
    'alpha': 0.6057645620141668
}

print("Setup complete.")

--- Using PROVEN Hyperparameters from winner_v1_301 ---
Setup complete.


In [2]:
# Block 2: Advanced Feature Engineering on Steroids
# ==============================================================================
print("--- Starting Block 2: Feature Engineering on Steroids ---")

# (This block is the same as the one from Supercharged_v2)
def feature_engineer_steroids(df):
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            df[f'{NUMS[i]}_x_{NUMS[j]}'] = df[NUMS[i]] * df[NUMS[j]]
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['year'] = df['sale_date'].dt.year
    df['month'] = df['sale_date'].dt.month
    df['weekofyear'] = df['sale_date'].dt.isocalendar().week.astype(int)
    df['dayofyear'] = df['sale_date'].dt.dayofyear
    df['quarter'] = df['sale_date'].dt.quarter
    df['year_diff'] = df['year'] - df['year_built']
    df['year_reno_diff'] = df['year'] - df['year_reno'].replace(0, np.nan)
    coords = df[['latitude', 'longitude']].copy()
    kmeans = MiniBatchKMeans(n_clusters=20, random_state=RANDOM_STATE, batch_size=256, n_init=10)
    df['geo_cluster'] = kmeans.fit_predict(coords)
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in df.columns: df[c] = np.log1p(df[c])
    return df

train_ids = df_train.index
test_ids = df_test.index
df_train['is_train'] = 1
df_test['is_train'] = 0
all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
all_data = feature_engineer_steroids(all_data)
cols_to_drop_final = ['sale_date']
all_data = all_data.drop(columns=cols_to_drop_final)
text_cols_impute = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
for col in all_data.columns:
    if col in text_cols_impute or all_data[col].dtype == 'object':
        all_data[col] = all_data[col].fillna('missing')
    else:
        all_data[col] = all_data[col].fillna(0)
X = all_data[all_data['is_train'] == 1].drop(columns=['is_train', 'sale_price'])
X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train', 'sale_price'])
X.index, X_test.index = train_ids, test_ids
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()

--- Starting Block 2: Feature Engineering on Steroids ---

Synthesized FE complete. Total features: 74


20

In [3]:
# Block 3: Training the PROVEN Model on BETTER Features
# ==============================================================================
print("\n--- STAGE 1: K-Fold Training of Mean Model (Elite Hybrid) ---")

grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_mean_preds, test_mean_preds = np.zeros(len(X)), np.zeros(len(X_test))
text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]
    X_train_fold, X_val_fold, X_test_fold = X_train.copy(), X_val.copy(), X_test.copy()

    # TF-IDF Features
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128)
        X_train_fold[col], X_val_fold[col], X_test_fold[col] = (X_train_fold[col].astype(str), X_val_fold[col].astype(str), X_test_fold[col].astype(str))
        tfidf.fit(X_train_fold[col])
        train_tfidf_matrix, val_tfidf_matrix, test_tfidf_matrix = (tfidf.transform(X_train_fold[col]).toarray(), tfidf.transform(X_val_fold[col]).toarray(), tfidf.transform(X_test_fold[col]).toarray())
        num_features = train_tfidf_matrix.shape[1]
        train_tfidf, val_tfidf, test_tfidf = (pd.DataFrame(train_tfidf_matrix, index=X_train_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(val_tfidf_matrix, index=X_val_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(test_tfidf_matrix, index=X_test_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]))
        X_train_fold, X_val_fold, X_test_fold = (pd.concat([X_train_fold, train_tfidf], axis=1), pd.concat([X_val_fold, val_tfidf], axis=1), pd.concat([X_test_fold, test_tfidf], axis=1))
    X_train_fold, X_val_fold, X_test_fold = (X_train_fold.drop(columns=text_cols), X_val_fold.drop(columns=text_cols), X_test_fold.drop(columns=text_cols))
    
    # Train Model using Native API and PROVEN parameters
    dtrain, dval, dtest = (xgb.DMatrix(X_train_fold, label=y_train), xgb.DMatrix(X_val_fold, label=y_val), xgb.DMatrix(X_test_fold))
    model = xgb.train(params=best_params_mean, dtrain=dtrain, num_boost_round=2500, evals=[(dval, 'eval')], early_stopping_rounds=100, verbose_eval=False)
    
    oof_mean_preds[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration))
    test_mean_preds += model.predict(dtest, iteration_range=(0, model.best_iteration)) / N_SPLITS
    print(f"Fold {fold+1} OOF RMSE: {np.sqrt(mean_squared_error(y_val, oof_mean_preds[val_idx])):,.2f}")

final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Stage 1 (Mean) Final OOF RMSE: ${final_mean_rmse:,.2f}")


--- STAGE 1: K-Fold Training of Mean Model (Elite Hybrid) ---

--- Fold 1/5 ---
Fold 1 OOF RMSE: 100,206.19

--- Fold 2/5 ---
Fold 2 OOF RMSE: 99,099.00

--- Fold 3/5 ---
Fold 3 OOF RMSE: 98,960.06

--- Fold 4/5 ---
Fold 4 OOF RMSE: 99,034.76

--- Fold 5/5 ---
Fold 5 OOF RMSE: 99,477.17

# Stage 1 (Mean) Final OOF RMSE: $99,356.50


In [4]:
# Block 4: Stage 2 - K-Fold Training of Error Model (Elite Hybrid)
# ==============================================================================
print("\n--- STAGE 2: K-Fold Training of Error Model (Elite Hybrid) ---")

# New target is the absolute error from the new mean model
error_target = np.abs(y_true - oof_mean_preds)

# Add OOF mean predictions as a feature for the error model
X_for_error = X.copy()
X_for_error['mean_pred_oof'] = oof_mean_preds
X_test_for_error = X_test.copy()
X_test_for_error['mean_pred_oof'] = test_mean_preds

# Initialize prediction arrays
oof_error_preds = np.zeros(len(X))
test_error_preds = np.zeros(len(X_test))

# Re-use the same folds for consistency
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X_for_error.iloc[train_idx], X_for_error.iloc[val_idx]
    y_train, y_val = error_target.iloc[train_idx], error_target.iloc[val_idx]

    # --- TF-IDF Features ---
    X_train_fold, X_val_fold, X_test_fold = X_train.copy(), X_val.copy(), X_test_for_error.copy()
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128)
        X_train_fold[col], X_val_fold[col], X_test_fold[col] = (X_train_fold[col].astype(str), X_val_fold[col].astype(str), X_test_fold[col].astype(str))
        tfidf.fit(X_train_fold[col])
        train_tfidf_matrix, val_tfidf_matrix, test_tfidf_matrix = (tfidf.transform(X_train_fold[col]).toarray(), tfidf.transform(X_val_fold[col]).toarray(), tfidf.transform(X_test_fold[col]).toarray())
        num_features = train_tfidf_matrix.shape[1]
        train_tfidf, val_tfidf, test_tfidf = (pd.DataFrame(train_tfidf_matrix, index=X_train_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(val_tfidf_matrix, index=X_val_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(test_tfidf_matrix, index=X_test_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]))
        X_train_fold, X_val_fold, X_test_fold = (pd.concat([X_train_fold, train_tfidf], axis=1), pd.concat([X_val_fold, val_tfidf], axis=1), pd.concat([X_test_fold, test_tfidf], axis=1))
    X_train_fold, X_val_fold, X_test_fold = (X_train_fold.drop(columns=text_cols), X_val_fold.drop(columns=text_cols), X_test_fold.drop(columns=text_cols))
    
    # --- Train Model using Native API and PROVEN error parameters ---
    dtrain, dval, dtest = (xgb.DMatrix(X_train_fold, label=y_train), xgb.DMatrix(X_val_fold, label=y_val), xgb.DMatrix(X_test_fold))
    model = xgb.train(params=best_params_error, dtrain=dtrain, num_boost_round=2000, evals=[(dval, 'eval')], early_stopping_rounds=100, verbose_eval=False)

    oof_error_preds[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration))
    test_error_preds += model.predict(dtest, iteration_range=(0, model.best_iteration)) / N_SPLITS
    print(f"Fold {fold+1} OOF Error RMSE: {np.sqrt(mean_squared_error(y_val, oof_error_preds[val_idx])):,.2f}")
    gc.collect()

final_error_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\n# Stage 2 (Error) Final OOF RMSE: ${final_error_rmse:,.2f}")


--- STAGE 2: K-Fold Training of Error Model (Elite Hybrid) ---

--- Fold 1/5 ---
Fold 1 OOF Error RMSE: 63,515.10

--- Fold 2/5 ---
Fold 2 OOF Error RMSE: 61,603.57

--- Fold 3/5 ---
Fold 3 OOF Error RMSE: 63,117.11

--- Fold 4/5 ---
Fold 4 OOF Error RMSE: 62,449.37

--- Fold 5/5 ---
Fold 5 OOF Error RMSE: 63,648.44

# Stage 2 (Error) Final OOF RMSE: $62,871.27


In [5]:
# Block 5: Final Asymmetric Calibration and Submission
# ==============================================================================
print("\n--- Final Asymmetric Calibration ---")

def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)

# Clip the OOF error predictions to be non-negative
oof_error_final = np.clip(oof_error_preds, 0, None)

best_a, best_b, best_metric = 2.0, 2.0, float('inf')

print("Starting grid search for optimal multipliers...")
# Define a wide search space to find the new optimum
search_space_a = np.arange(1.80, 2.31, 0.01)
search_space_b = np.arange(2.00, 2.51, 0.01)

for a in search_space_a:
    for b in search_space_b:
        low = oof_mean_preds - oof_error_final * a
        high = oof_mean_preds + oof_error_final * b
        
        metric, coverage = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA, return_coverage=True)
        
        if metric < best_metric:
            best_metric = metric
            best_a, best_b = a, b
            print(f"New best score: {best_metric:,.2f} | Coverage: {coverage:.3f} | a={best_a:.2f}, b={best_b:.2f}")

print(f"\nGrid search complete. Final OOF Score: {best_metric:,.2f}. Best multipliers: a={best_a:.2f}, b={best_b:.2f}")

# --- Create Final Submission ---
print("\nCreating final submission file...")
test_error_final = np.clip(test_error_preds, 0, None)

final_lower = test_mean_preds - test_error_final * best_a
final_upper = test_mean_preds + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper) # Ensure upper is always > lower

# Reload original test set to get the 'id' column
submission_df = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])
submission_df['pi_lower'] = final_lower
submission_df['pi_upper'] = final_upper

submission_df.to_csv('submission_elite_hybrid_v1.csv', index=False)
print("\n'submission_elite_hybrid_v1.csv' created successfully!")
display(submission_df.head())


--- Final Asymmetric Calibration ---
Starting grid search for optimal multipliers...
New best score: 306,859.32 | Coverage: 0.872 | a=1.80, b=2.00
New best score: 306,732.56 | Coverage: 0.873 | a=1.80, b=2.01
New best score: 306,612.63 | Coverage: 0.874 | a=1.80, b=2.02
New best score: 306,499.73 | Coverage: 0.875 | a=1.80, b=2.03
New best score: 306,394.47 | Coverage: 0.875 | a=1.80, b=2.04
New best score: 306,297.52 | Coverage: 0.876 | a=1.80, b=2.05
New best score: 306,209.87 | Coverage: 0.877 | a=1.80, b=2.06
New best score: 306,129.37 | Coverage: 0.877 | a=1.80, b=2.07
New best score: 306,057.18 | Coverage: 0.878 | a=1.80, b=2.08
New best score: 305,992.82 | Coverage: 0.879 | a=1.80, b=2.09
New best score: 305,936.12 | Coverage: 0.879 | a=1.80, b=2.10
New best score: 305,886.00 | Coverage: 0.880 | a=1.80, b=2.11
New best score: 305,843.16 | Coverage: 0.881 | a=1.80, b=2.12
New best score: 305,808.14 | Coverage: 0.881 | a=1.80, b=2.13
New best score: 305,780.15 | Coverage: 0.882 |

,id,pi_lower,pi_upper
0,200000,823353.915322,1.057924e+06
1,200001,604000.723828,8.270034e+05
2,200002,434027.889863,6.545598e+05
3,200003,277207.624336,3.992899e+05
4,200004,290914.192793,8.439127e+05


In [11]:
# Block 6: Data Preparation for PyTorch (Corrected with Target Scaling)
# ==============================================================================
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

print("\n--- Preparing Data for PyTorch Model ---")

# --- Feature Scaling ---
X_torch = X.drop(columns=text_cols, errors='ignore').copy()
X_test_torch = X_test.drop(columns=text_cols, errors='ignore').copy()
feature_scaler = MinMaxScaler()
X_torch_scaled = feature_scaler.fit_transform(X_torch)
X_test_torch_scaled = feature_scaler.transform(X_test_torch)

# === FIX: Scale the target variable (y_true) ===
target_scaler = MinMaxScaler()
y_true_scaled = target_scaler.fit_transform(y_true.values.reshape(-1, 1))

# --- Tensor Conversion ---
X_tensor = torch.tensor(X_torch_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_true_scaled, dtype=torch.float32) # Already has the right shape
X_test_tensor = torch.tensor(X_test_torch_scaled, dtype=torch.float32)

# --- Dataloaders ---
BATCH_SIZE = 1024
train_dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataset = TensorDataset(X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nData successfully scaled and converted to PyTorch Tensors.")


--- Preparing Data for PyTorch Model ---

Data successfully scaled and converted to PyTorch Tensors.


In [35]:
# Block 7: Defining the DEEP Champion MDN Model
# ==============================================================================
print("--- Defining DEEP Champion PyTorch MDN Model ---")

# === FIX: Made the network deeper and wider for more learning capacity ===
HIDDEN_SIZE_1 = 512 # Wider
HIDDEN_SIZE_2 = 256 # Wider
HIDDEN_SIZE_3 = 128 # New deeper layer
N_GAUSSIANS = 7   # More "experts" to capture complex distributions

class MDN(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, hidden_size_3, n_gaussians):
        super(MDN, self).__init__()
        self.n_gaussians = n_gaussians
        
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size_1),
            nn.BatchNorm1d(hidden_size_1),
            nn.ReLU(),
            nn.Dropout(0.25), # Slightly more dropout for the bigger network
            nn.Linear(hidden_size_1, hidden_size_2),
            nn.BatchNorm1d(hidden_size_2),
            nn.ReLU(),
            nn.Dropout(0.25),
            # New third hidden layer
            nn.Linear(hidden_size_2, hidden_size_3),
            nn.BatchNorm1d(hidden_size_3),
            nn.ReLU(),
            nn.Dropout(0.25),
        )
        
        # Output layers now connect from the new deeper layer
        self.pi_logits = nn.Linear(hidden_size_3, n_gaussians)
        self.mu = nn.Linear(hidden_size_3, n_gaussians)
        self.log_sigma = nn.Linear(hidden_size_3, n_gaussians)

    def forward(self, x):
        x = self.network(x)
        log_pi = nn.functional.log_softmax(self.pi_logits(x), dim=-1)
        mu = self.mu(x)
        log_sigma = self.log_sigma(x)
        return log_pi, mu, log_sigma

# (The stable loss function remains the same)
def mdn_loss_fn_stable(log_pi, mu, log_sigma, y):
    log_sigma = torch.clamp(log_sigma, min=-10.0, max=10.0)
    log_gauss_prob = -0.5 * ((y - mu)**2 * torch.exp(-2 * log_sigma) + 2 * log_sigma + math.log(2 * math.pi))
    log_likelihood = torch.logsumexp(log_pi + log_gauss_prob, dim=-1)
    return -torch.mean(log_likelihood)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

--- Defining DEEP Champion PyTorch MDN Model ---
Using device: cuda


In [37]:
# Block 8: AGGRESSIVE Training of the DEEP Champion MDN
# ==============================================================================
from tqdm import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau

print("--- Starting AGGRESSIVE MDN Model Training ---")

# --- Initialize a fresh model for this final run ---
# === FIX: Added the new HIDDEN_SIZE_3 argument to the model constructor ===
model_mdn_final = MDN(
    INPUT_SIZE, 
    HIDDEN_SIZE_1, 
    HIDDEN_SIZE_2, 
    HIDDEN_SIZE_3, # The missing argument
    N_GAUSSIANS
).to(device)

optimizer_final = optim.Adam(model_mdn_final.parameters(), lr=0.001)

N_EPOCHS_FINAL = 75 
scheduler_final = ReduceLROnPlateau(optimizer_final, mode='min', factor=0.5, patience=5)

model_mdn_final.train()
for epoch in range(N_EPOCHS_FINAL):
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS_FINAL}", leave=False)
    for batch_X, batch_y in progress_bar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer_final.zero_grad()
        log_pi, mu, log_sigma = model_mdn_final(batch_X)
        loss = mdn_loss_fn_stable(log_pi, mu, log_sigma, batch_y)
        if torch.isnan(loss): 
            print(f"NaN loss detected at epoch {epoch+1}. Stopping training.")
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mdn_final.parameters(), 1.0)
        optimizer_final.step()
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    if torch.isnan(loss): break
    
    avg_epoch_loss = epoch_loss / len(train_loader)
    scheduler_final.step(avg_epoch_loss)
    print(f"Epoch {epoch+1} complete. Average Loss: {avg_epoch_loss:.4f}")

print("\n--- Final Model Training Complete ---")

--- Starting AGGRESSIVE MDN Model Training ---


Epoch 1 complete. Average Loss: -0.9523


Epoch 2 complete. Average Loss: -1.6387


Epoch 3 complete. Average Loss: -1.7344


Epoch 4 complete. Average Loss: -1.7929


...

Epoch 71 complete. Average Loss: -2.3166


Epoch 72 complete. Average Loss: -2.3282


Epoch 73 complete. Average Loss: -2.3206


Epoch 74 complete. Average Loss: -2.3237


Epoch 75 complete. Average Loss: -2.3202

--- Final Model Training Complete ---


In [38]:
# Block 9: Find Optimal Quantiles for the BOLDER Model
# ==============================================================================
print("\n--- Finding Optimal Quantiles for Final Submission Model ---")

# First, get predictions from our new final model on the TRAIN set
# This is our OOF equivalent for finding the best quantiles
train_full_dataset = TensorDataset(X_tensor)
train_full_loader = DataLoader(train_full_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_mdn_final.eval()
log_pi_oof, mu_oof, log_sigma_oof = [], [], []
with torch.no_grad():
    for batch_X_tuple in tqdm(train_full_loader, desc="Getting OOF predictions"):
        batch_X = batch_X_tuple[0].to(device)
        log_pi, mu, log_sigma = model_mdn_final(batch_X)
        log_pi_oof.append(log_pi.cpu().numpy())
        mu_oof.append(mu.cpu().numpy())
        log_sigma_oof.append(log_sigma.cpu().numpy())

pi_oof = np.exp(np.concatenate(log_pi_oof, axis=0))
mu_oof = np.concatenate(mu_oof, axis=0)
sigma_oof = np.exp(np.concatenate(log_sigma_oof, axis=0))

# Now, perform the calibration search on these OOF predictions
best_score, best_lower_q, best_upper_q = float('inf'), 0, 0
search_space_lower = np.arange(0.01, 0.11, 0.01)
search_space_upper = np.arange(0.90, 0.99, 0.01)
quantile_preds_oof = {q: gmm_quantile(pi_oof, mu_oof, sigma_oof, q) for q in np.unique(np.concatenate([search_space_lower, search_space_upper]))}

for lq in search_space_lower:
    for uq in search_space_upper:
        if uq <= lq: continue
        lower_b = target_scaler.inverse_transform(quantile_preds_oof[lq].reshape(-1, 1)).flatten()
        upper_b = target_scaler.inverse_transform(quantile_preds_oof[uq].reshape(-1, 1)).flatten()
        score, _ = winkler_score(y_true.values, lower_b, upper_b, return_coverage=True)
        if score < best_score:
            best_score = score
            best_lower_q, best_upper_q = lq, uq
            print(f"New best OOF score: {score:,.2f} with quantiles L={lq:.2f}, U={uq:.2f}")

print(f"\nOPTIMAL QUANTILES FOUND: Lower={best_lower_q:.2f}, Upper={best_upper_q:.2f}")

# Block 10: Generate Final Submission with Optimal Quantiles
# ==============================================================================
print("\n--- Generating FINAL Submission with Bolder Model and Optimal Quantiles ---")

# Get predictions for the actual TEST set
model_mdn_final.eval()
log_pi_test, mu_test, log_sigma_test = [], [], []
with torch.no_grad():
    for batch_X_tuple in tqdm(test_loader, desc="Getting Test predictions"):
        batch_X = batch_X_tuple[0].to(device)
        log_pi, mu, log_sigma = model_mdn_final(batch_X)
        log_pi_test.append(log_pi.cpu().numpy())
        mu_test.append(mu.cpu().numpy())
        log_sigma_test.append(log_sigma.cpu().numpy())

pi_test = np.exp(np.concatenate(log_pi_test, axis=0))
mu_test = np.concatenate(mu_test, axis=0)
sigma_test = np.exp(np.concatenate(log_sigma_test, axis=0))

# Calculate the final bounds using our OPTIMAL quantiles
final_lower_scaled = gmm_quantile(pi_test, mu_test, sigma_test, best_lower_q)
final_upper_scaled = gmm_quantile(pi_test, mu_test, sigma_test, best_upper_q)

# Inverse transform to get final dollar values
final_lower = target_scaler.inverse_transform(final_lower_scaled.reshape(-1, 1)).flatten()
final_upper = target_scaler.inverse_transform(final_upper_scaled.reshape(-1, 1)).flatten()
final_lower = np.nan_to_num(final_lower, nan=np.median(np.nan_to_num(final_lower)))
final_upper = np.nan_to_num(final_upper, nan=np.median(np.nan_to_num(final_upper)))



--- Finding Optimal Quantiles for Final Submission Model ---


Calculating 98th percentile: 100%|████████████████████████████████████████████| 200000/200000 [00:57<00:00, 3487.33it/s]


New best OOF score: 471,912.22 with quantiles L=0.01, U=0.90
New best OOF score: 461,923.55 with quantiles L=0.01, U=0.91
New best OOF score: 452,968.34 with quantiles L=0.01, U=0.92
New best OOF score: 445,260.58 with quantiles L=0.01, U=0.93
New best OOF score: 439,121.47 with quantiles L=0.01, U=0.94
New best OOF score: 435,213.63 with quantiles L=0.01, U=0.95
New best OOF score: 434,564.69 with quantiles L=0.01, U=0.96
New best OOF score: 429,151.47 with quantiles L=0.02, U=0.92
New best OOF score: 421,443.71 with quantiles L=0.02, U=0.93
New best OOF score: 415,304.59 with quantiles L=0.02, U=0.94
New best OOF score: 411,396.76 with quantiles L=0.02, U=0.95
New best OOF score: 410,747.81 with quantiles L=0.02, U=0.96
New best OOF score: 408,928.94 with quantiles L=0.03, U=0.93
New best OOF score: 402,789.83 with quantiles L=0.03, U=0.94
New best OOF score: 398,881.99 with quantiles L=0.03, U=0.95
New best OOF score: 398,233.05 with quantiles L=0.03, U=0.96
New best OOF score: 394,

Calculating 96th percentile: 100%|████████████████████████████████████████████| 200000/200000 [00:49<00:00, 4013.54it/s]


In [ ]:

# Create submission file
submission_df = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])
submission_df['pi_lower'] = final_lower
submission_df['pi_upper'] = np.maximum(final_lower, final_upper) + 1e-3
submission_df.to_csv('submission_pytorch_mdn_WINNER.csv', index=False)
print("\n'submission_pytorch_mdn_WINNER.csv' created successfully!")
display(submission_df.head())

In [24]:
# Block 10: Splitting Data for Local Validation
# ==============================================================================
from sklearn.model_selection import train_test_split

print("--- Splitting Data for Local Validation ---")

# We will use an 80/20 split. 80% for training, 20% for validation.
X_train_val, X_val_val, y_train_val, y_val_val = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=RANDOM_STATE
)

# We also need the original, unscaled y-values for the final score calculation
_, _, y_train_orig_val, y_val_orig_val = train_test_split(
    X_tensor, y_true, test_size=0.2, random_state=RANDOM_STATE
)


# Create new DataLoaders for this validation training run
train_dataset_val = TensorDataset(X_train_val, y_train_val)
train_loader_val = DataLoader(train_dataset_val, batch_size=BATCH_SIZE, shuffle=True)

print("Validation data split successfully.")
print(f"Validation Training Shape:   {X_train_val.shape}")
print(f"Validation Hold-out Shape:   {X_val_val.shape}")

--- Splitting Data for Local Validation ---
Validation data split successfully.
Validation Training Shape:   torch.Size([160000, 68])
Validation Hold-out Shape:   torch.Size([40000, 68])


In [25]:
# Block 11: Re-Training Model on the 80% Validation Split
# ==============================================================================
print("--- Re-training a fresh MDN model for validation ---")

# Initialize a completely new model and optimizer for this run
model_mdn_val = MDN(INPUT_SIZE, HIDDEN_SIZE_1, HIDDEN_SIZE_2, N_GAUSSIANS).to(device)
optimizer_val = optim.Adam(model_mdn_val.parameters(), lr=0.001)
scheduler_val = ReduceLROnPlateau(optimizer_val, mode='min', factor=0.5, patience=3)

N_EPOCHS_VAL = 30 # Use the same number of epochs

model_mdn_val.train()
for epoch in range(N_EPOCHS_VAL):
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader_val, desc=f"Epoch {epoch+1}/{N_EPOCHS_VAL}", leave=False)
    for batch_X, batch_y in progress_bar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer_val.zero_grad()
        log_pi, mu, log_sigma = model_mdn_val(batch_X)
        loss = mdn_loss_fn_stable(log_pi, mu, log_sigma, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mdn_val.parameters(), 1.0)
        optimizer_val.step()
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_epoch_loss = epoch_loss / len(train_loader_val)
    scheduler_val.step(avg_epoch_loss)

print("\n--- Validation Model Training Complete ---")

--- Re-training a fresh MDN model for validation ---



--- Validation Model Training Complete ---


In [26]:
# Block 12: Score Calculation on Validation Set
# ==============================================================================
print("--- Scoring MDN model on the hold-out validation set ---")

model_mdn_val.eval()
with torch.no_grad():
    # Predict on the validation hold-out set
    log_pi_val, mu_val, log_sigma_val = model_mdn_val(X_val_val.to(device))
    
    # Move predictions to CPU and convert to numpy
    pi_val = torch.exp(log_pi_val).cpu().numpy()
    mu_val = mu_val.cpu().numpy()
    sigma_val = torch.exp(log_sigma_val).cpu().numpy()

# Calculate the quantiles for the validation predictions
lower_bound_scaled_val = gmm_quantile(pi_val, mu_val, sigma_val, 0.05)
upper_bound_scaled_val = gmm_quantile(pi_val, mu_val, sigma_val, 0.95)

# Inverse transform the predictions back to dollar amounts
final_lower_val = target_scaler.inverse_transform(lower_bound_scaled_val.reshape(-1, 1)).flatten()
final_upper_val = target_scaler.inverse_transform(upper_bound_scaled_val.reshape(-1, 1)).flatten()

# Use our original Winkler score function from the XGBoost notebooks
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)

# Calculate the final score on the un-scaled validation labels
mdn_score, mdn_coverage = winkler_score(y_val_orig_val.values, final_lower_val, final_upper_val, return_coverage=True)

print("\n--- LOCAL VALIDATION SCORE ---")
print(f"PyTorch MDN Score: {mdn_score:,.2f}")
print(f"PyTorch MDN Coverage: {mdn_coverage:.3f}")
print("------------------------------")
print(f"For comparison, your best XGBoost OOF score was: 304,352.95")
print("------------------------------")

if mdn_score < 304352.95:
    print("\nSUCCESS! The PyTorch MDN model is better than the XGBoost model.")
else:
    print("\nCLOSE! The PyTorch MDN is not yet better. Further tuning of the NN architecture or training epochs may be needed.")

--- Scoring MDN model on the hold-out validation set ---


Calculating 95th percentile: 100%|██████████████████████████████████████████████| 40000/40000 [00:11<00:00, 3400.61it/s]


--- LOCAL VALIDATION SCORE ---
PyTorch MDN Score: 476,408.50
PyTorch MDN Coverage: 0.949
------------------------------
For comparison, your best XGBoost OOF score was: 304,352.95
------------------------------

CLOSE! The PyTorch MDN is not yet better. Further tuning of the NN architecture or training epochs may be needed.


In [27]:
# Block 13: MDN Quantile Calibration Search
# ==============================================================================
print("\n--- Starting MDN Quantile Calibration Search ---")

best_score = float('inf')
best_lower_q = 0
best_upper_q = 0
best_coverage = 0

# Define a search space for the lower and upper quantiles
# Let's search around the 0.05 and 0.95 marks
search_space_lower = np.arange(0.02, 0.08, 0.01)
search_space_upper = np.arange(0.92, 0.98, 0.01)

# Pre-calculate all the quantiles we might need to speed up the search
# This avoids re-calculating the same quantile multiple times
print("Pre-calculating quantiles for grid search...")
needed_quantiles = np.unique(np.concatenate([search_space_lower, search_space_upper]))
quantile_predictions = {}
for q in needed_quantiles:
    quantile_predictions[q] = gmm_quantile(pi_val, mu_val, sigma_val, q)

print("Grid search starting...")
for lower_q in search_space_lower:
    for upper_q in search_space_upper:
        # Get the pre-calculated, scaled quantile predictions
        lower_bound_scaled_val = quantile_predictions[lower_q]
        upper_bound_scaled_val = quantile_predictions[upper_q]
        
        # Inverse transform back to dollar amounts
        final_lower_val = target_scaler.inverse_transform(lower_bound_scaled_val.reshape(-1, 1)).flatten()
        final_upper_val = target_scaler.inverse_transform(upper_bound_scaled_val.reshape(-1, 1)).flatten()
        
        # Calculate the Winkler score for this pair of quantiles
        score, coverage = winkler_score(y_val_orig_val.values, final_lower_val, final_upper_val, return_coverage=True)
        
        # Check if this is the best score so far
        if score < best_score:
            best_score = score
            best_lower_q = lower_q
            best_upper_q = upper_q
            best_coverage = coverage
            print(f"New best MDN score: {best_score:,.2f} | Coverage: {best_coverage:.3f} | Lower q: {best_lower_q:.2f}, Upper q: {best_upper_q:.2f}")

print("\n--- MDN CALIBRATION COMPLETE ---")
print(f"Final Best MDN Score: {best_score:,.2f}")
print(f"Achieved with Optimal Quantiles: Lower={best_lower_q:.2f}, Upper={best_upper_q:.2f}")
print(f"Resulting Coverage: {best_coverage:.3f}")
print("---------------------------------")
print(f"For comparison, your best XGBoost OOF score was: 304,352.95")


--- Starting MDN Quantile Calibration Search ---
Pre-calculating quantiles for grid search...


Calculating 97th percentile: 100%|██████████████████████████████████████████████| 40000/40000 [00:09<00:00, 4050.90it/s]

Grid search starting...
New best MDN score: 487,249.85 | Coverage: 0.955 | Lower q: 0.02, Upper q: 0.92
New best MDN score: 471,039.21 | Coverage: 0.950 | Lower q: 0.03, Upper q: 0.92
New best MDN score: 461,892.11 | Coverage: 0.943 | Lower q: 0.04, Upper q: 0.92
New best MDN score: 456,653.52 | Coverage: 0.935 | Lower q: 0.05, Upper q: 0.92
New best MDN score: 453,988.47 | Coverage: 0.927 | Lower q: 0.06, Upper q: 0.92
New best MDN score: 453,239.29 | Coverage: 0.919 | Lower q: 0.07, Upper q: 0.92

--- MDN CALIBRATION COMPLETE ---
Final Best MDN Score: 453,239.29
Achieved with Optimal Quantiles: Lower=0.07, Upper=0.92
Resulting Coverage: 0.919
---------------------------------
For comparison, your best XGBoost OOF score was: 304,352.95
